# 🥬 LettuceDetect × RAGTruth: Hallucination Detection Training & Evaluation

This notebook trains LettuceDetect on the **RAGTruth** dataset and produces:
- **Accuracy vs Epoch** graph
- **Training & Validation Loss Curve** graph

**Make sure to select GPU runtime**: Runtime → Change runtime type → T4 GPU

### ⚠️ IMPORTANT: Run Cell 1 FIRST, restart runtime, then run all cells from Cell 2

## 1️⃣ Install Dependencies (Run FIRST, then restart runtime)

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 1: Run this cell FIRST, then restart runtime
#         Runtime → Restart runtime
#         Then SKIP this cell and run from Cell 2 onwards
# ═══════════════════════════════════════════════════════════
!pip install --force-reinstall numpy torch transformers -q
!pip install scikit-learn tqdm matplotlib seaborn -q
!pip install datasets -q

# Clone LettuceDetect
import os
if not os.path.exists('LettuceDetect'):
    !git clone https://github.com/KRLabsOrg/LettuceDetect.git

%cd LettuceDetect
!pip install -e . --no-deps -q

print("\n" + "="*50)
print("✅ All packages installed!")
print("")
print("👉 NOW: Go to Runtime → Restart runtime")
print("👉 THEN: Skip this cell, run all cells from Cell 2")
print("="*50)

## 2️⃣ Verify Setup (Run AFTER restart)

In [ ]:
# Navigate to LettuceDetect directory
import os
if os.path.exists('LettuceDetect'):
    os.chdir('LettuceDetect')
print(f"Working dir: {os.getcwd()}")

import torch
import numpy as np
import transformers
print(f"numpy: {np.__version__}")
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

# Test imports
from transformers import AutoTokenizer, AutoModelForTokenClassification
from lettucedetect.datasets.hallucination_dataset import HallucinationData, HallucinationDataset
print("\n✅ All imports working!")

## 3️⃣ Download & Preprocess RAGTruth Dataset

In [ ]:
import json
import random
from pathlib import Path
from dataclasses import dataclass

# Patch HallucinationSample to accept 'ragtruth' dataset
import lettucedetect.datasets.hallucination_dataset as hd

@dataclass
class HallucinationSample:
    prompt: str
    answer: str
    labels: list
    split: str
    task_type: str
    dataset: str
    language: str

    def to_json(self):
        return {"prompt": self.prompt, "answer": self.answer, "labels": self.labels,
                "split": self.split, "task_type": self.task_type,
                "dataset": self.dataset, "language": self.language}

    @classmethod
    def from_json(cls, json_dict):
        return cls(prompt=json_dict["prompt"], answer=json_dict["answer"],
                   labels=json_dict["labels"], split=json_dict["split"],
                   task_type=json_dict["task_type"], dataset=json_dict["dataset"],
                   language=json_dict["language"])

hd.HallucinationSample = HallucinationSample
print("✅ Patched HallucinationSample")

In [ ]:
from datasets import load_dataset

print("📥 Downloading RAGTruth dataset from HuggingFace...")
ds = load_dataset("lytang/RAGTruth")
print(f"✅ Downloaded! Splits: {list(ds.keys())}")
for split in ds:
    print(f"  {split}: {len(ds[split])} samples")

In [ ]:
from lettucedetect.datasets.hallucination_dataset import HallucinationData

random.seed(42)
os.makedirs("data/ragtruth", exist_ok=True)

all_samples = []
MAX_SAMPLES = 2000  # Cap per split to avoid OOM on Colab

for split_name in ds:
    split_data = list(ds[split_name])
    # Map HF split names to our convention
    if 'train' in split_name:
        our_split = 'train'
    elif 'val' in split_name or 'dev' in split_name:
        our_split = 'dev'
    else:
        our_split = 'test'

    if len(split_data) > MAX_SAMPLES:
        random.shuffle(split_data)
        split_data = split_data[:MAX_SAMPLES]

    for item in split_data:
        # Extract prompt/context
        prompt = item.get('source', '') or item.get('prompt', '') or item.get('context', '')
        answer = item.get('response', '') or item.get('answer', '') or item.get('output', '')
        task_type = item.get('task_type', 'unknown') or item.get('type', 'unknown')

        if not prompt or not answer:
            continue

        # Build labels from annotations
        labels = []
        annotations = item.get('labels', []) or item.get('annotations', []) or []

        if isinstance(annotations, list):
            for ann in annotations:
                if isinstance(ann, dict):
                    start = ann.get('start', 0)
                    end = ann.get('end', 0)
                    label_type = ann.get('label_type', '') or ann.get('label', '')
                    if start < end and label_type:
                        labels.append({'start': start, 'end': end, 'label': label_type})

        sample = HallucinationSample(
            prompt=prompt, answer=answer, labels=labels,
            split=our_split, task_type=str(task_type),
            dataset='ragtruth', language='en'
        )
        all_samples.append(sample)

# Stats
train_count = sum(1 for s in all_samples if s.split == 'train')
dev_count = sum(1 for s in all_samples if s.split == 'dev')
test_count = sum(1 for s in all_samples if s.split == 'test')
hall_count = sum(1 for s in all_samples if len(s.labels) > 0)

print(f"\n📊 Total samples: {len(all_samples)}")
print(f"   Train: {train_count} | Dev: {dev_count} | Test: {test_count}")
print(f"   Hallucinated: {hall_count} | Supported: {len(all_samples) - hall_count}")

# Save processed data
hall_data = HallucinationData(samples=all_samples)
with open('data/ragtruth/ragtruth_data.json', 'w') as f:
    json.dump(hall_data.to_json(), f, indent=2)
print("✅ Saved to data/ragtruth/ragtruth_data.json")

## 4️⃣ Prepare DataLoaders

In [ ]:
import json, random, numpy as np, torch
from pathlib import Path
from torch.utils.data import DataLoader
from transformers import AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification
from lettucedetect.datasets.hallucination_dataset import HallucinationData, HallucinationDataset

random.seed(123); np.random.seed(123); torch.manual_seed(123); torch.cuda.manual_seed_all(123)

# Load data
hall_data = HallucinationData.from_json(
    json.loads(Path('data/ragtruth/ragtruth_data.json').read_text())
)

# Split samples
train_samples = [s for s in hall_data.samples if s.split == 'train']
dev_samples = [s for s in hall_data.samples if s.split == 'dev']
test_samples = [s for s in hall_data.samples if s.split == 'test']

# If no dev split exists, carve out 10% from train
if len(dev_samples) == 0:
    random.shuffle(train_samples)
    dev_size = int(len(train_samples) * 0.1)
    dev_samples = train_samples[-dev_size:]
    train_samples = train_samples[:-dev_size]

print(f"Train: {len(train_samples)}, Dev: {len(dev_samples)}, Test: {len(test_samples)}")

# Setup model & tokenizer
MODEL_NAME = 'answerdotai/ModernBERT-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
collator = DataCollatorForTokenClassification(tokenizer=tokenizer, label_pad_token_id=-100)

train_ds = HallucinationDataset(train_samples, tokenizer)
dev_ds = HallucinationDataset(dev_samples, tokenizer)

BS = 8
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, collate_fn=collator)
dev_loader = DataLoader(dev_ds, batch_size=BS, shuffle=False, collate_fn=collator)

print(f"\n✅ DataLoaders ready | Train batches: {len(train_loader)} | Dev batches: {len(dev_loader)}")

## 5️⃣ Train the Model 🚀 (Custom Loop with Metrics Tracking)

In [ ]:
import time
from datetime import timedelta
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

# ═══════════════════════════════════════════════════════════
# Hyperparameters
# ═══════════════════════════════════════════════════════════
EPOCHS = 6
LR = 1e-5
OUTPUT = 'output/ragtruth_detector'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=2, trust_remote_code=True
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# ═══════════════════════════════════════════════════════════
# Tracking lists for graphs
# ═══════════════════════════════════════════════════════════
train_losses = []      # avg training loss per epoch
val_losses = []        # avg validation loss per epoch
train_accuracies = []  # token-level accuracy on train per epoch
val_accuracies = []    # token-level accuracy on val per epoch
best_f1 = 0.0

print(f"\n🚀 Training on {device} | {len(train_loader)} steps/epoch × {EPOCHS} epochs")
print(f"   Model: {MODEL_NAME}")
print(f"   Batch size: {BS} | Learning rate: {LR}")
print("=" * 60)

start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    print(f"\n📌 Epoch {epoch + 1}/{EPOCHS}")

    # ─── Training Phase ───
    model.train()
    total_train_loss = 0
    num_train_batches = 0
    all_train_preds = []
    all_train_labels = []

    progress = tqdm(train_loader, desc="  Training", leave=True)
    for batch in progress:
        optimizer.zero_grad()
        outputs = model(
            batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            labels=batch['labels'].to(device)
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        num_train_batches += 1

        # Collect predictions for accuracy
        preds = torch.argmax(outputs.logits, dim=-1)
        mask = batch['labels'] != -100
        all_train_preds.extend(preds[mask].cpu().numpy().tolist())
        all_train_labels.extend(batch['labels'][mask].cpu().numpy().tolist())

        progress.set_postfix({
            'loss': f'{loss.item():.4f}',
            'avg': f'{total_train_loss / num_train_batches:.4f}'
        })

    avg_train_loss = total_train_loss / num_train_batches
    train_acc = accuracy_score(all_train_labels, all_train_preds)
    train_losses.append(avg_train_loss)
    train_accuracies.append(train_acc)

    # ─── Validation Phase ───
    model.eval()
    total_val_loss = 0
    num_val_batches = 0
    all_val_preds = []
    all_val_labels = []
    all_val_probs = []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc="  Validating", leave=True):
            outputs = model(
                batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
                labels=batch['labels'].to(device)
            )
            total_val_loss += outputs.loss.item()
            num_val_batches += 1

            preds = torch.argmax(outputs.logits, dim=-1)
            mask = batch['labels'] != -100
            all_val_preds.extend(preds[mask].cpu().numpy().tolist())
            all_val_labels.extend(batch['labels'][mask].cpu().numpy().tolist())

    avg_val_loss = total_val_loss / num_val_batches
    val_acc = accuracy_score(all_val_labels, all_val_preds)
    val_losses.append(avg_val_loss)
    val_accuracies.append(val_acc)

    # Compute F1 for model saving
    from sklearn.metrics import precision_recall_fscore_support
    _, _, f1, _ = precision_recall_fscore_support(
        all_val_labels, all_val_preds, labels=[0, 1], average=None, zero_division=0
    )
    val_f1_hall = f1[1]  # hallucination class F1

    epoch_time = time.time() - epoch_start
    print(f"  ⏱️  Epoch {epoch+1} done in {timedelta(seconds=int(epoch_time))}")
    print(f"  📉 Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"  🎯 Train Acc:  {train_acc:.4f} | Val Acc:  {val_acc:.4f}")
    print(f"  🔍 Val F1 (Hallucination): {val_f1_hall:.4f}")

    # Save best model
    if val_f1_hall > best_f1:
        best_f1 = val_f1_hall
        os.makedirs(OUTPUT, exist_ok=True)
        model.save_pretrained(OUTPUT)
        tokenizer.save_pretrained(OUTPUT)
        print(f"  💾 New best F1: {best_f1:.4f} — model saved to '{OUTPUT}'")

    print("-" * 60)

total_time = time.time() - start_time
print(f"\n✅ Training complete in {timedelta(seconds=int(total_time))}")
print(f"🏆 Best Hallucination F1: {best_f1:.4f}")

## 6️⃣ Visualization 📈 — Accuracy vs Epoch & Loss Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.titlesize': 15,
    'axes.labelsize': 13,
})

epochs_range = list(range(1, EPOCHS + 1))
os.makedirs('output', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('LettuceDetect on RAGTruth — Training Metrics', fontsize=18, fontweight='bold', y=1.03)

# ━━━ Plot 1: Accuracy vs Epoch ━━━
ax = axes[0]
ax.plot(epochs_range, train_accuracies, 'o-', color='#2ecc71', linewidth=2.5,
        markersize=8, label='Train Accuracy', zorder=3)
ax.plot(epochs_range, val_accuracies, 's-', color='#e74c3c', linewidth=2.5,
        markersize=8, label='Validation Accuracy', zorder=3)
ax.fill_between(epochs_range, train_accuracies, alpha=0.1, color='#2ecc71')
ax.fill_between(epochs_range, val_accuracies, alpha=0.1, color='#e74c3c')

# Annotate final values
ax.annotate(f'{train_accuracies[-1]:.4f}',
            xy=(epochs_range[-1], train_accuracies[-1]),
            xytext=(10, 10), textcoords='offset points',
            fontsize=10, color='#2ecc71', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=1.5))
ax.annotate(f'{val_accuracies[-1]:.4f}',
            xy=(epochs_range[-1], val_accuracies[-1]),
            xytext=(10, -15), textcoords='offset points',
            fontsize=10, color='#e74c3c', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5))

ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs Epoch')
ax.set_xticks(epochs_range)
ax.legend(loc='lower right', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_ylim(0, 1.05)

# ━━━ Plot 2: Training & Validation Loss Curve ━━━
ax = axes[1]
ax.plot(epochs_range, train_losses, 'o-', color='#3498db', linewidth=2.5,
        markersize=8, label='Training Loss', zorder=3)
ax.plot(epochs_range, val_losses, 's-', color='#e67e22', linewidth=2.5,
        markersize=8, label='Validation Loss', zorder=3)
ax.fill_between(epochs_range, train_losses, alpha=0.1, color='#3498db')
ax.fill_between(epochs_range, val_losses, alpha=0.1, color='#e67e22')

# Annotate final values
ax.annotate(f'{train_losses[-1]:.4f}',
            xy=(epochs_range[-1], train_losses[-1]),
            xytext=(10, 10), textcoords='offset points',
            fontsize=10, color='#3498db', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#3498db', lw=1.5))
ax.annotate(f'{val_losses[-1]:.4f}',
            xy=(epochs_range[-1], val_losses[-1]),
            xytext=(10, -15), textcoords='offset points',
            fontsize=10, color='#e67e22', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#e67e22', lw=1.5))

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training & Validation Loss Curve')
ax.set_xticks(epochs_range)
ax.legend(loc='upper right', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('output/ragtruth_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved to output/ragtruth_training_curves.png")

## 7️⃣ Evaluation on Test Set 📊

In [ ]:
from sklearn.metrics import (
    accuracy_score, classification_report, precision_recall_fscore_support,
    roc_curve, auc, confusion_matrix
)

# Load best model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_model = AutoModelForTokenClassification.from_pretrained(OUTPUT, trust_remote_code=True).to(device)
best_tokenizer = AutoTokenizer.from_pretrained(OUTPUT)
collator = DataCollatorForTokenClassification(tokenizer=best_tokenizer, label_pad_token_id=-100)

# Prepare test data
hall_data = HallucinationData.from_json(
    json.loads(Path('data/ragtruth/ragtruth_data.json').read_text())
)
test_samples = [s for s in hall_data.samples if s.split == 'test']
print(f"📦 Test samples: {len(test_samples)}")
print(f"   Hallucinated: {sum(1 for s in test_samples if s.labels)}")
print(f"   Supported: {sum(1 for s in test_samples if not s.labels)}")

test_loader = DataLoader(
    HallucinationDataset(test_samples, best_tokenizer),
    batch_size=8, shuffle=False, collate_fn=collator
)

# ━━━ Token-Level Evaluation ━━━
best_model.eval()
tok_preds, tok_labels, tok_probs = [], [], []
ex_preds, ex_labels, ex_probs = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        out = best_model(
            batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device)
        )
        probs = torch.softmax(out.logits, dim=-1)
        preds = torch.argmax(out.logits, dim=-1)
        mask = batch['labels'] != -100

        tok_preds.extend(preds[mask].cpu().numpy().tolist())
        tok_labels.extend(batch['labels'][mask].cpu().numpy().tolist())
        tok_probs.extend(probs[:, :, 1][mask].cpu().numpy().tolist())

        # Example-level
        for i in range(batch['labels'].size(0)):
            sl = batch['labels'][i]; sp = preds[i].cpu(); vm = sl != -100
            if vm.sum() == 0:
                ex_labels.append(0); ex_preds.append(0); ex_probs.append(0.0)
            else:
                sv = sl[vm].cpu(); pv = sp[vm]; ppv = probs[i][vm]
                ex_labels.append(1 if (sv == 1).any().item() else 0)
                ex_preds.append(1 if (pv == 1).any().item() else 0)
                ex_probs.append(ppv[:, 1].max().item())

# ━━━ Token-Level Metrics ━━━
pt, rt, f1t, _ = precision_recall_fscore_support(tok_labels, tok_preds, labels=[0, 1], average=None, zero_division=0)
acc_t = accuracy_score(tok_labels, tok_preds)
fpr_t, tpr_t, _ = roc_curve(tok_labels, tok_probs); auroc_t = auc(fpr_t, tpr_t)
cm_t = confusion_matrix(tok_labels, tok_preds, labels=[0, 1])

print("\n" + "=" * 60 + "\n📊 TOKEN-LEVEL RESULTS\n" + "=" * 60)
print(classification_report(tok_labels, tok_preds, target_names=['Supported', 'Hallucinated'], digits=4, zero_division=0))
print(f"Accuracy: {acc_t:.4f}  |  AUROC: {auroc_t:.4f}")

# ━━━ Example-Level Metrics ━━━
pe, re, f1e, _ = precision_recall_fscore_support(ex_labels, ex_preds, labels=[0, 1], average=None, zero_division=0)
acc_e = accuracy_score(ex_labels, ex_preds)
fpr_e, tpr_e, _ = roc_curve(ex_labels, ex_probs); auroc_e = auc(fpr_e, tpr_e)
cm_e = confusion_matrix(ex_labels, ex_preds, labels=[0, 1])

print("\n" + "=" * 60 + "\n📊 EXAMPLE-LEVEL RESULTS\n" + "=" * 60)
print(classification_report(ex_labels, ex_preds, target_names=['Supported', 'Hallucinated'], digits=4, zero_division=0))
print(f"Accuracy: {acc_e:.4f}  |  AUROC: {auroc_e:.4f}")
print("\n✅ Evaluation done!")

## 8️⃣ Full Visualization Dashboard 📈

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle('LettuceDetect on RAGTruth — Complete Performance Dashboard',
             fontsize=18, fontweight='bold', y=1.02)

# ━━━ 1. Accuracy vs Epoch ━━━
ax = axes[0][0]
ax.plot(epochs_range, train_accuracies, 'o-', color='#2ecc71', lw=2.5, ms=8, label='Train Accuracy')
ax.plot(epochs_range, val_accuracies, 's-', color='#e74c3c', lw=2.5, ms=8, label='Val Accuracy')
ax.fill_between(epochs_range, train_accuracies, alpha=0.1, color='#2ecc71')
ax.fill_between(epochs_range, val_accuracies, alpha=0.1, color='#e74c3c')
ax.set(xlabel='Epoch', ylabel='Accuracy', title='Accuracy vs Epoch')
ax.set_xticks(epochs_range)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3, linestyle='--')

# ━━━ 2. Training & Validation Loss ━━━
ax = axes[0][1]
ax.plot(epochs_range, train_losses, 'o-', color='#3498db', lw=2.5, ms=8, label='Train Loss')
ax.plot(epochs_range, val_losses, 's-', color='#e67e22', lw=2.5, ms=8, label='Val Loss')
ax.fill_between(epochs_range, train_losses, alpha=0.1, color='#3498db')
ax.fill_between(epochs_range, val_losses, alpha=0.1, color='#e67e22')
ax.set(xlabel='Epoch', ylabel='Loss', title='Training & Validation Loss')
ax.set_xticks(epochs_range)
ax.legend(loc='upper right'); ax.grid(True, alpha=0.3, linestyle='--')

# ━━━ 3. ROC Curves ━━━
ax = axes[1][0]
ax.plot(fpr_t, tpr_t, color='#e74c3c', lw=2.5, label=f'Token AUC={auroc_t:.4f}')
ax.plot(fpr_e, tpr_e, color='#3498db', lw=2.5, label=f'Example AUC={auroc_e:.4f}')
ax.plot([0, 1], [0, 1], '--', color='#bdc3c7', lw=1.5)
ax.fill_between(fpr_t, tpr_t, alpha=0.08, color='#e74c3c')
ax.fill_between(fpr_e, tpr_e, alpha=0.08, color='#3498db')
ax.set(xlabel='FPR', ylabel='TPR', title='ROC Curves')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)

# ━━━ 4. Confusion Matrix ━━━
ax = axes[1][1]
sns.heatmap(cm_e, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Supported', 'Hallucinated'],
            yticklabels=['Supported', 'Hallucinated'],
            ax=ax, annot_kws={'size': 14})
ax.set(xlabel='Predicted', ylabel='Actual', title='Confusion Matrix (Example-Level)')

plt.tight_layout()
plt.savefig('output/ragtruth_full_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved to output/ragtruth_full_dashboard.png")

In [ ]:
# ━━━ FINAL SUMMARY ━━━
print("\n" + "=" * 65)
print("   🥬 LETTUCEDETECT × RAGTRUTH — PERFORMANCE SUMMARY")
print("=" * 65)
print(f"\n{'Metric':<28} {'Token-Level':>12} {'Example-Level':>14}")
print("-" * 58)
print(f"{'Hallucinated F1':<28} {f1t[1]:>12.4f} {f1e[1]:>14.4f}")
print(f"{'Hallucinated Precision':<28} {pt[1]:>12.4f} {pe[1]:>14.4f}")
print(f"{'Hallucinated Recall':<28} {rt[1]:>12.4f} {re[1]:>14.4f}")
print(f"{'Supported F1':<28} {f1t[0]:>12.4f} {f1e[0]:>14.4f}")
print(f"{'Accuracy':<28} {acc_t:>12.4f} {acc_e:>14.4f}")
print(f"{'AUROC':<28} {auroc_t:>12.4f} {auroc_e:>14.4f}")
print("=" * 58)

# Save results
results = {
    'training_history': {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
    },
    'token_level': {
        'supported': {'p': float(pt[0]), 'r': float(rt[0]), 'f1': float(f1t[0])},
        'hallucinated': {'p': float(pt[1]), 'r': float(rt[1]), 'f1': float(f1t[1])},
        'accuracy': float(acc_t), 'auroc': float(auroc_t)
    },
    'example_level': {
        'supported': {'p': float(pe[0]), 'r': float(re[0]), 'f1': float(f1e[0])},
        'hallucinated': {'p': float(pe[1]), 'r': float(re[1]), 'f1': float(f1e[1])},
        'accuracy': float(acc_e), 'auroc': float(auroc_e)
    },
}
with open('output/ragtruth_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\n✅ Results saved to output/ragtruth_results.json")

## 9️⃣ Download Results

In [ ]:
!zip -r ragtruth_results.zip output/ragtruth_detector output/ragtruth_results.json output/ragtruth_training_curves.png output/ragtruth_full_dashboard.png
from google.colab import files
files.download('ragtruth_results.zip')
print("📥 Downloaded!")